# End-to-end: injecting errors and measuring quality

The full experiment setup on one measure: learn a reference from a clean table, corrupt a test table with
[`tab_err`](https://github.com/calgo-lab/tab_err), measure it, and check the measurement against the
injected ground truth.

## 1. Imports

In [1]:
import narwhals as nw
import numpy as np
import polars as pl
from tab_err import error_type
from tab_err.api import high_level

from dqmeasure import DataAccuracyRange

## 2. Generate the clean (train) table

In [2]:
def make_sensor_readings(n: int = 200, seed: int = 0) -> pl.DataFrame:
    rng = np.random.default_rng(seed)
    return pl.DataFrame(
        {
            "temperature": rng.uniform(0, 100, n).round(2),  # required range 0..100 C
            "humidity": rng.uniform(20, 80, n).round(2),  # required range 20..80 %
            "pressure": rng.uniform(950, 1050, n).round(2),  # required range 950..1050 hPa
        }
    )


train = make_sensor_readings(n=200, seed=0)  # clean instance == training data
train.head()

temperature,humidity,pressure
f64,f64,f64
63.7,39.18,970.22
26.98,31.25,1043.79
4.1,60.35,959.48
1.65,31.71,950.49
81.33,54.66,982.29


## 3. Fit the measures on the clean data

Every measure is scoped to **one column**, so we build one `DataAccuracyRange` per
sensor column. `fit` learns the reference interval `[min, max]` from the clean data.

In [3]:
measures = {col: DataAccuracyRange(col).fit(train) for col in train.columns}
{col: (m.low_, m.high_) for col, m in measures.items()}  # learned intervals

{'temperature': (0.27, 99.72),
 'humidity': (20.02, 79.82),
 'pressure': (950.49, 1049.95)}

## 4. Build the dirty (test) table

We inject **Outliers** with `tab_err`. It returns the corrupted table and a boolean `error_mask` that indicates which cells are corrupted.

In [4]:
test_clean = make_sensor_readings(n=100, seed=1)

dirty_pd, mask_pd = high_level.create_errors(
    test_clean.to_pandas(),
    error_rate=0.10,  # corrupt approx. 10% of cells per column
    error_types_to_include=[error_type.Outlier()],
    seed=42,
)

dirty = pl.from_pandas(dirty_pd)
error_mask = pl.from_pandas(mask_pd)
dirty.head()

temperature,humidity,pressure
f64,f64,f64
51.18,59.23,1006.21
95.05,45.87,988.78
-83.024586,72.04,1029.17
94.86,57.93,1010.51
31.18,68.62,1036.13


## 5. `predict`

`predict` returns the measurement as a series aligned with the input rows: `1.0` if the
value is inside the learned interval, `0.0` if outside, and null if the value is missing.

In [5]:
cells = {col: m.predict(dirty) for col, m in measures.items()}
cells["temperature"].head()

temperature
f64
1.0
1.0
0.0
1.0
1.0
1.0
1.0
1.0
1.0


## 6. `score`

`score` aggregates the cell measure into the ISO measurement function `X = A / B` (in-range count / non-null count). With a 10 % error rate we
expect roughly 0.90.

In [6]:
scores = {col: m.score(dirty) for col, m in measures.items()}
scores

{'temperature': 0.9, 'humidity': 0.89, 'pressure': 0.9}

## 7. Validation against the ground-truth

Because `tab_err` returns an `error_mask` with the positions of the corrupted cells, we can check how well the measure recovers them.

In [7]:
print(f"{'column':12s} {'injected':>8s} {'flagged':>8s} {'precision':>9s} {'recall':>7s}")
for col in measures:
    flagged = [v == 0.0 for v in nw.from_native(cells[col], series_only=True).to_list()]
    truth = error_mask[col].to_list()
    tp = sum(f and t for f, t in zip(flagged, truth, strict=True))
    fp = sum(f and not t for f, t in zip(flagged, truth, strict=True))
    fn = sum((not f) and t for f, t in zip(flagged, truth, strict=True))
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    print(f"{col:12s} {sum(truth):8d} {sum(flagged):8d} {precision:9.2f} {recall:7.2f}")

column       injected  flagged precision  recall
temperature        10       10      1.00    1.00
humidity           10       11      0.91    1.00
pressure           10       10      1.00    1.00


Works well. Occasional false positives are expected, because a *clean* test value can fall just outside the `[min, max]`
we learned on the training sample.